# Simulación de eventos discretos de un proceso de servicio con SimPy

## Caso: atención de solicitudes administrativas

Una mesa de servicio recibe solicitudes durante una jornada de 8 horas. Las solicitudes pueden ser constancias, aclaraciones de pago, actualización de datos o recepción de documentos. La coordinación desea decidir cuántas personas servidoras asignar.

Se comparan tres alternativas:

- una persona servidora;
- dos personas servidoras;
- tres personas servidoras.

La pregunta central es: ¿qué capacidad mantiene una espera razonable, reduce pendientes al cierre y usa responsablemente los recursos?

El notebook emplea datos sintéticos, pero con unidades y supuestos realistas. La estructura puede recalibrarse después con registros históricos.

## tl;dr

La simulación de eventos discretos representa el proceso como una secuencia de acontecimientos: llegan solicitudes, esperan si el personal está ocupado, reciben atención y salen del sistema.

Se compararán:

- espera promedio, mediana y percentil 90;
- espera máxima;
- solicitudes generadas y atendidas;
- pendientes al cierre;
- utilización del personal;
- comportamiento de la acumulación durante la jornada.

La recomendación se basará en calidad de servicio y eficiencia. No se debe elegir solo la alternativa con menor espera: también importa el número de recursos y el margen para absorber variaciones.

## 1. Contexto y supuestos del proceso

La regla de atención será FIFO: primero en llegar, primero en ser atendido. La jornada tendrá 8 horas, equivalentes a 480 minutos.

| Elemento | Supuesto |
|---|---|
| Jornada | 480 minutos |
| Tiempo promedio entre llegadas | 6 minutos |
| Llegadas esperadas | 80 solicitudes por jornada |
| Tiempo mínimo de atención | 4 minutos |
| Tiempo más probable | 8 minutos |
| Tiempo máximo | 15 minutos |
| Alternativas | 1, 2 y 3 personas servidoras |
| Réplicas | 30 jornadas por alternativa |

El tiempo entre llegadas se modelará con una distribución exponencial: las solicitudes son irregulares, pero tienen un promedio conocido.

El servicio se modelará con una distribución triangular. Es útil cuando conocemos un mínimo, un valor típico y un máximo. Su media teórica es:

(4 + 8 + 15) / 3 = 9 minutos

Una persona tendría una capacidad aproximada de 480 / 9 = 53.3 atenciones por jornada. La demanda esperada es 480 / 6 = 80 solicitudes. Esta comparación anticipa presión sobre una sola persona y justifica evaluar alternativas.

## 2. Técnica: simulación de eventos discretos

A diferencia de un cálculo estático, la simulación de eventos discretos conserva el orden temporal del proceso. El reloj salta de un evento importante al siguiente.

Eventos del caso:

1. llega una solicitud;
2. entra a la cola si todas las personas están ocupadas;
3. inicia la atención;
4. termina el servicio;
5. sale del sistema.

Conceptos de SimPy:

- Environment: reloj y agenda de eventos.
- Resource: recurso con capacidad limitada.
- request: petición de una plaza de atención.
- yield: pausa hasta que ocurra un evento.
- timeout: duración de una actividad.
- env.now: tiempo actual del reloj.
- process: actividad que SimPy administra.

Cada solicitud es un proceso independiente. Si no hay una persona disponible, queda esperando; cuando una plaza se libera, SimPy continúa el proceso.

Esta técnica es apropiada para servicios porque permite observar colas, saturación y experiencia individual, no solamente promedios.

## 3. Instalación e importación

Se instala SimPy para que el notebook pueda ejecutarse en un entorno nuevo de Google Colab.

Se cargan también NumPy para aleatoriedad, pandas para tablas, Matplotlib y seaborn para gráficos, y dataclass para registrar cada solicitud.

La semilla aleatoria permitirá repetir el ejercicio y obtener los mismos resultados.

In [ ]:
!pip install simpy -q

import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dataclasses import dataclass, asdict
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("SimPy y librerías cargadas correctamente.")

### Explicación sencilla del bloque de instalación

Este bloque prepara el espacio de trabajo. SimPy es la herramienta que hará avanzar el reloj del proceso y coordinará las solicitudes que llegan, esperan y reciben servicio.

NumPy se usa para generar variación realista. En la vida real, dos días no tienen exactamente las mismas llegadas ni los mismos tiempos de atención. Pandas organiza cada resultado como una tabla. Matplotlib y seaborn convierten esos resultados en gráficos que permiten explicar el comportamiento a personas no técnicas.

La semilla no elimina la aleatoriedad del modelo; únicamente permite repetir la misma experiencia de simulación. Esto es útil para revisar resultados, enseñar el método y comparar cambios de manera justa.

## 4. Parámetros y unidades

Todos los tiempos se expresan en minutos. Mantener una unidad común evita errores como mezclar una llegada de 6 minutos con una jornada expresada en horas.

La distribución triangular queda limitada entre 4 y 15 minutos, por lo que nunca produce tiempos negativos o imposibles para este proceso.

In [ ]:
SEMILLA_BASE = 2026
MEDIA_INTERARRIBO = 6.0
SERVICIO_MIN = 4.0
SERVICIO_MODA = 8.0
SERVICIO_MAX = 15.0
DURACION_JORNADA = 480.0
REPETICIONES = 30

media_servicio = (SERVICIO_MIN + SERVICIO_MODA + SERVICIO_MAX) / 3

print(f"Media teórica del servicio: {media_servicio:.2f} minutos")
print(f"Demanda esperada: {DURACION_JORNADA / MEDIA_INTERARRIBO:.1f} solicitudes")
print(f"Capacidad aproximada por persona: {DURACION_JORNADA / media_servicio:.1f} solicitudes")

### Explicación sencilla de los parámetros

Esta celda funciona como el panel de control del experimento. Si se cambia el promedio entre llegadas, se está diciendo que la demanda del servicio cambió. Si se cambia el mínimo, el valor típico o el máximo del servicio, se está diciendo que el trámite se volvió más rápido o más complejo.

La unidad común es el minuto. La jornada de 480 minutos equivale exactamente a 8 horas. El promedio de servicio es 9 minutos, por lo que una persona puede atender aproximadamente 53 solicitudes en una jornada si estuviera ocupada todo el tiempo. La demanda promedio es de aproximadamente 80 solicitudes.

Esta comparación es importante: una sola persona no puede atender 80 solicitudes si en promedio solo tiene capacidad para unas 53. Eso hace razonable esperar una cola grande en ese escenario. Dos personas tendrían una capacidad teórica aproximada de 107 atenciones y, por tanto, dispondrían de más margen.

## 5. Validación visual de las entradas

Antes de construir la cola, se generan 10,000 observaciones de cada distribución.

La distribución de llegadas debe tener media cercana a 6 minutos. La distribución de servicio debe quedar entre 4 y 15 minutos y concentrarse cerca de 8.

Este paso verifica que los datos de entrada hacen sentido antes de interpretar resultados del proceso.

In [ ]:
rng = np.random.default_rng(SEMILLA_BASE)

interarribos = rng.exponential(MEDIA_INTERARRIBO, size=10_000)
servicios = rng.triangular(
    SERVICIO_MIN, SERVICIO_MODA, SERVICIO_MAX, size=10_000
)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(interarribos, bins=50, kde=True, color="#2563EB", ax=ax[0])
ax[0].axvline(MEDIA_INTERARRIBO, color="black", linestyle="--")
ax[0].set_title("Tiempos entre llegadas")
ax[0].set_xlabel("Minutos entre solicitudes")
ax[0].set_ylabel("Frecuencia")

sns.histplot(servicios, bins=35, kde=True, color="#10B981", ax=ax[1])
ax[1].axvline(media_servicio, color="black", linestyle="--")
ax[1].set_title("Tiempos de servicio")
ax[1].set_xlabel("Minutos por atención")
ax[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

print(f"Media observada entre llegadas: {interarribos.mean():.2f} minutos")
print(f"Media observada de servicio: {servicios.mean():.2f} minutos")
print(f"Rango del servicio: {servicios.min():.2f} a {servicios.max():.2f} minutos")

### Explicación sencilla de las distribuciones

El histograma de llegadas representa el tiempo que pasa entre una solicitud y la siguiente. Un valor pequeño significa que llegaron dos solicitudes con poca separación; un valor grande significa que hubo un periodo de calma. Ambos comportamientos son normales en una oficina de atención.

El histograma de servicio representa la duración de cada atención. La distribución está limitada entre 4 y 15 minutos, por lo que no aparecen duraciones negativas ni tiempos imposibles. El valor más probable es 8 minutos, que funciona como una atención típica.

La media observada puede no ser exactamente igual a la media configurada en una muestra finita. Lo importante es que sea razonablemente cercana. Las diferencias pequeñas son parte de la variación aleatoria, no necesariamente un error.

### Interpretación de las entradas

La gráfica de llegadas debe mostrar muchos intervalos cortos y algunos largos. Esto representa irregularidad: no todas las personas llegan con la misma separación.

La gráfica del servicio debe estar completamente dentro de 4 y 15 minutos, con mayor concentración cerca de 8. En términos operativos, una atención sencilla tarda alrededor de 4 minutos, una típica 8 y una compleja hasta 15.

Si estas entradas fueran absurdas, la simulación podría ejecutarse sin error pero sus conclusiones no serían confiables.

## 6. Registro de solicitudes

Cada solicitud tendrá:

- identificador;
- minuto de llegada;
- inicio y fin del servicio;
- espera;
- duración de atención;
- tiempo total en el sistema;
- estado.

El registro permite construir métricas, tablas y visualizaciones después de ejecutar el modelo.

In [ ]:
@dataclass
class RegistroSolicitud:
    solicitud_id: int
    llegada: float
    inicio_servicio: float
    salida: float
    espera: float
    servicio: float
    tiempo_sistema: float
    estado: str

### Explicación sencilla del registro

Este bloque define una ficha para cada solicitud. Es como una bitácora: registra cuándo llegó la persona, cuándo comenzó a ser atendida, cuánto esperó, cuánto duró el trámite y cuándo salió.

Tener esta ficha permite responder preguntas concretas. Por ejemplo, una solicitud que llegó en el minuto 30 y empezó en el minuto 42 esperó 12 minutos. Si su atención duró 7 minutos, permaneció 19 minutos en el sistema.

La columna estado permite distinguir entre trabajo terminado y trabajo que quedó pendiente al cierre. Esta separación es importante para no confundir una solicitud que todavía sigue en proceso con una solicitud ya atendida.

## 7. Proceso de atención

La función representa la vida de una solicitud.

Primero se registra la llegada. Después solicita el recurso de atención. Si todas las personas están ocupadas, el proceso queda pausado en la cola.

Cuando obtiene el recurso, se registra el inicio y se calcula la espera. Luego se genera un tiempo de servicio triangular, avanza el reloj durante esa actividad y se registra la salida.

La instrucción with libera automáticamente el recurso al terminar la atención.

In [ ]:
def atender_solicitud(env, solicitud_id, servidores, registros, llegadas, rng):
    llegada = env.now
    llegadas.append({"solicitud_id": solicitud_id, "llegada": llegada})

    with servidores.request() as recurso:
        yield recurso

        inicio_servicio = env.now
        espera = inicio_servicio - llegada

        servicio = rng.triangular(
            SERVICIO_MIN, SERVICIO_MODA, SERVICIO_MAX
        )

        yield env.timeout(servicio)

        salida = env.now
        estado = (
            "atendida"
            if salida <= DURACION_JORNADA
            else "pendiente_al_cierre"
        )

        registros.append(
            RegistroSolicitud(
                solicitud_id=solicitud_id,
                llegada=llegada,
                inicio_servicio=inicio_servicio,
                salida=salida,
                espera=espera,
                servicio=servicio,
                tiempo_sistema=salida - llegada,
                estado=estado
            )
        )

### Explicación sencilla del proceso de atención

Esta función es la traducción del proceso real al lenguaje de simulación. La solicitud no desaparece ni recibe atención instantánea por defecto. Primero intenta obtener un lugar con una persona servidora.

Cuando todas las personas están ocupadas, SimPy deja la solicitud en espera. Cuando una persona termina, la siguiente solicitud puede comenzar. Por eso la espera surge naturalmente de la relación entre demanda y capacidad.

El tiempo de servicio se genera para cada solicitud, de modo que algunas atenciones son cortas, otras típicas y otras más largas. El modelo no supone que todos los trámites tarden lo mismo, porque eso no sería realista.

## 8. Generador de llegadas

El generador crea solicitudes mientras el reloj esté dentro de la jornada.

La cantidad exacta de llegadas no se fija. Se genera de manera aleatoria a partir del promedio de 6 minutos. Esto representa mejor un servicio real, donde un día puede recibir más solicitudes y otro menos.

Cada solicitud se registra como un proceso separado para que SimPy pueda coordinar varias atenciones en paralelo.

In [ ]:
def generar_llegadas(env, servidores, registros, llegadas, rng, media_interarribo):
    solicitud_id = 0
    siguiente_llegada = 0.0

    while siguiente_llegada < DURACION_JORNADA:
        yield env.timeout(siguiente_llegada - env.now)

        solicitud_id += 1
        env.process(
            atender_solicitud(
                env, solicitud_id, servidores, registros, llegadas, rng
            )
        )

        siguiente_llegada += rng.exponential(media_interarribo)

### Explicación sencilla del generador de llegadas

Este bloque representa el flujo de personas usuarias. La primera solicitud llega al inicio y las siguientes aparecen de acuerdo con tiempos entre llegadas aleatorios.

No se fuerza una cantidad exacta de 80 solicitudes. El valor de 80 es el promedio esperado, pero una jornada real puede tener 74, 82 o 90. Permitir esa variación hace que el ejercicio sea más parecido a la operación.

Cada llegada genera un proceso independiente. SimPy puede así mantener varias solicitudes esperando mientras otras están siendo atendidas.

## 9. Simular una jornada

Se crea un ambiente nuevo por jornada. Así no se mezclan solicitudes entre réplicas.

El parámetro capacity indica cuántas personas pueden atender simultáneamente. Con capacidad 2, dos solicitudes pueden estar en servicio al mismo tiempo.

El modelo se ejecuta hasta 480 minutos, que es el cierre de la jornada.

In [ ]:
def simular_jornada(
    numero_servidores,
    media_interarribo=MEDIA_INTERARRIBO,
    semilla=SEMILLA_BASE
):
    rng = np.random.default_rng(semilla)
    env = simpy.Environment()
    servidores = simpy.Resource(
        env, capacity=numero_servidores
    )
    registros = []
    llegadas = []

    env.process(
        generar_llegadas(
            env, servidores, registros, llegadas, rng, media_interarribo
        )
    )

    env.run(until=DURACION_JORNADA)

    datos = pd.DataFrame([asdict(x) for x in registros])
    llegadas_df = pd.DataFrame(llegadas)

    if datos.empty:
        datos = llegadas_df.copy()
        datos["inicio_servicio"] = np.nan
        datos["salida"] = np.nan
        datos["espera"] = np.nan
        datos["servicio"] = np.nan
        datos["tiempo_sistema"] = np.nan
        datos["estado"] = "pendiente_al_cierre"
    else:
        atendidos_ids = set(datos["solicitud_id"])
        faltantes = llegadas_df[~llegadas_df["solicitud_id"].isin(atendidos_ids)].copy()
        if not faltantes.empty:
            faltantes["inicio_servicio"] = np.nan
            faltantes["salida"] = np.nan
            faltantes["espera"] = np.nan
            faltantes["servicio"] = np.nan
            faltantes["tiempo_sistema"] = np.nan
            faltantes["estado"] = "pendiente_al_cierre"
            datos = pd.concat([datos, faltantes], ignore_index=True)

    return datos.sort_values("solicitud_id").reset_index(drop=True)

### Explicación sencilla de una jornada

Aquí se construye el escenario completo. El ambiente contiene el reloj. El recurso contiene el número de personas servidoras. El generador produce llegadas y el ambiente ejecuta los eventos hasta el minuto 480.

Crear un ambiente nuevo en cada jornada es equivalente a comenzar un día nuevo. No se arrastra una cola anterior de manera accidental, salvo que el modelo se amplíe explícitamente para representar esa situación.

El resultado es una tabla de solicitudes. Esa tabla es la base para calcular espera, atención, pendientes y utilización.

## 10. Primera inspección de resultados

Se ejecuta una jornada con dos personas servidoras.

La comprobación temporal valida que:

tiempo en el sistema = espera + servicio

El error máximo debe ser prácticamente cero. Esta revisión confirma que los cálculos internos son consistentes.

In [ ]:
datos_jornada = simular_jornada(2, semilla=SEMILLA_BASE)

display(datos_jornada.head(10))

error = (
    datos_jornada["tiempo_sistema"]
    - datos_jornada["espera"]
    - datos_jornada["servicio"]
).abs().max()

print(f"Solicitudes registradas: {len(datos_jornada)}")
print(f"Espera mínima: {datos_jornada['espera'].min():.2f} minutos")
print(f"Espera máxima: {datos_jornada['espera'].max():.2f} minutos")
print(f"Error máximo de consistencia: {error:.10f}")

### Explicación sencilla de la inspección inicial

Esta primera corrida sirve como prueba de comprensión. La tabla muestra casos reales dentro del escenario simulado: algunas solicitudes comienzan inmediatamente y otras tienen que esperar.

La comprobación de consistencia confirma que el tiempo total se calcula como espera más servicio. Si el error máximo es cero o muy cercano a cero, las columnas son coherentes entre sí.

Esta validación no demuestra que el modelo sea perfecto. Solo demuestra que la lógica de tiempos está bien conectada. La validez del modelo también depende de que los supuestos representen al proceso real.

### Interpretación

Las solicitudes con espera cercana a cero encontraron capacidad disponible. Una espera positiva significa que todas las personas estaban ocupadas al llegar.

Esta jornada sirve para comprender el mecanismo, pero no para decidir capacidad. Una sola secuencia aleatoria puede ser especialmente favorable o desfavorable.

## 11. Métricas operativas

Se calculan:

- generadas: llegadas durante la jornada;
- atendidas: servicios que terminaron antes del cierre;
- pendientes: solicitudes que no terminaron;
- espera promedio y mediana;
- percentil 90 de espera;
- espera máxima;
- servicio promedio;
- utilización.

El percentil 90 es relevante porque muestra la experiencia de las personas que enfrentan las mayores demoras.

La utilización es una aproximación: minutos de atención dividido entre la capacidad teórica total. No modela reuniones, pausas o tareas externas.

In [ ]:
def resumir_jornada(datos, servidores, duracion=DURACION_JORNADA):
    atendidas = datos[datos["salida"] <= duracion]
    pendientes = len(datos) - len(atendidas)

    if len(atendidas) == 0:
        return {
            "servidores": servidores,
            "generadas": len(datos),
            "atendidas": 0,
            "pendientes": pendientes,
            "espera_promedio": np.nan,
            "mediana_espera": np.nan,
            "p90_espera": np.nan,
            "espera_maxima": np.nan,
            "servicio_promedio": np.nan,
            "utilizacion_pct": 0.0
        }

    trabajo = atendidas["servicio"].sum()

    return {
        "servidores": servidores,
        "generadas": len(datos),
        "atendidas": len(atendidas),
        "pendientes": pendientes,
        "espera_promedio": atendidas["espera"].mean(),
        "mediana_espera": atendidas["espera"].median(),
        "p90_espera": atendidas["espera"].quantile(.90),
        "espera_maxima": atendidas["espera"].max(),
        "servicio_promedio": atendidas["servicio"].mean(),
        "utilizacion_pct": 100 * trabajo / (duracion * servidores)
    }

display(pd.DataFrame([resumir_jornada(datos_jornada, 2)]).round(2))

### Explicación sencilla de las métricas

La espera promedio es útil para conocer la experiencia general, pero puede ocultar casos muy tardados. Por eso también se calcula la mediana y el percentil 90.

Si el percentil 90 vale 15 minutos, aproximadamente nueve de cada diez solicitudes esperan 15 minutos o menos, mientras que el grupo restante espera más. Para evaluar un servicio, esta medida suele ser más informativa que mirar únicamente el promedio.

Los pendientes representan solicitudes que llegaron antes del cierre pero no terminaron dentro de la jornada. La utilización compara el trabajo de atención contra la capacidad disponible. Una utilización de 98% indica que el equipo opera casi al límite y tiene poco margen para absorber variación.

## 12. Visualización del flujo

El gráfico tipo Gantt muestra una fila por solicitud:

- azul: tiempo de espera;
- verde: tiempo de servicio;
- línea roja: cierre de la jornada.

Esta visualización ayuda a entender cómo los promedios surgen de experiencias individuales.

In [ ]:
def graficar_flujo(datos, titulo, limite=45):
    datos_plot = datos.sort_values("solicitud_id").head(limite)

    fig, ax = plt.subplots(figsize=(14, 8))

    for posicion, (_, fila) in enumerate(datos_plot.iterrows()):
        ax.barh(
            posicion, fila["espera"], left=fila["llegada"],
            color="#93C5FD", edgecolor="white",
            label="Espera" if posicion == 0 else ""
        )
        ax.barh(
            posicion, fila["servicio"],
            left=fila["inicio_servicio"],
            color="#34D399", edgecolor="white",
            label="Servicio" if posicion == 0 else ""
        )

    ax.axvline(
        DURACION_JORNADA, color="#DC2626",
        linestyle="--", linewidth=2, label="Cierre"
    )
    ax.set_title(titulo)
    ax.set_xlabel("Minutos desde el inicio de la jornada")
    ax.set_ylabel("Solicitudes en orden de llegada")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

graficar_flujo(
    datos_jornada,
    "Flujo de solicitudes con dos personas servidoras"
)

### Explicación sencilla del gráfico de flujo

Cada fila representa una solicitud. La parte azul es tiempo que la persona estuvo esperando. La parte verde es el tiempo en que recibió atención.

Este gráfico permite ver la diferencia entre una llegada y un inicio de atención. Si ambos momentos coinciden, no hubo espera. Si el inicio está más adelante, apareció una cola.

La línea roja es el cierre de la jornada, no una falla. Sirve para identificar si existe trabajo que continúa después del horario normal. El límite de 45 solicitudes solo mejora la lectura visual; no elimina registros del cálculo.

### Interpretación del flujo

Segmentos azules largos indican que una solicitud esperó antes de ser atendida. Si los segmentos azules se alargan conforme avanza la jornada, la cola está acumulándose.

El segmento verde es el trabajo real de atención. La línea roja representa el minuto 480. Un servicio que continúa después de esa línea representa trabajo pendiente para el siguiente periodo.

La gráfica se limita a 45 solicitudes para conservar legibilidad; las métricas incluyen todos los registros.

## 13. Réplicas y comparación de capacidad

Se ejecutan 30 jornadas por cada alternativa. Las semillas cambian entre réplicas para representar días diferentes, pero las distribuciones y la duración de jornada permanecen iguales.

Las réplicas permiten observar el valor promedio y la variabilidad. Esto evita recomendar una capacidad basándose en una sola jornada aleatoria.

In [ ]:
resultados = []

for servidores in [1, 2, 3]:
    for replica in range(REPETICIONES):
        datos = simular_jornada(
            servidores,
            semilla=SEMILLA_BASE + replica
        )
        fila = resumir_jornada(datos, servidores)
        fila["replica"] = replica + 1
        resultados.append(fila)

resultados = pd.DataFrame(resultados)

tabla = (
    resultados.groupby("servidores")
    .agg(
        generadas=("generadas", "mean"),
        atendidas=("atendidas", "mean"),
        pendientes=("pendientes", "mean"),
        espera_promedio=("espera_promedio", "mean"),
        mediana_espera=("mediana_espera", "mean"),
        p90_espera=("p90_espera", "mean"),
        espera_maxima=("espera_maxima", "mean"),
        utilizacion=("utilizacion_pct", "mean")
    )
    .reset_index()
)

display(tabla.round(2))

### Explicación sencilla de las réplicas

Una jornada simulada es como observar un solo día. Para tomar una decisión necesitamos observar muchos días posibles.

Las 30 réplicas cambian los momentos de llegada y los tiempos de atención. Al resumirlas, obtenemos una visión más estable del comportamiento promedio y también podemos observar días mejores y peores.

Comparar las mismas distribuciones con una, dos y tres personas permite estudiar el efecto de la capacidad. No se está cambiando el tipo de trámite; se está cambiando la cantidad de atención disponible.

## 14. Visualizaciones de escenarios

La espera promedio muestra la experiencia típica. El boxplot del percentil 90 muestra cómo cambia la cola larga entre jornadas.

La utilización se interpreta junto con la espera: alta utilización puede ser eficiente, pero si se acerca al límite también deja poco margen ante variaciones.

In [ ]:
tabla["cumple_promedio"] = tabla["espera_promedio"] < 5
tabla["cumple_p90"] = tabla["p90_espera"] < 15

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(
    data=tabla, x="servidores", y="espera_promedio",
    color="#2563EB", ax=ax[0]
)
ax[0].axhline(5, color="#DC2626", linestyle="--")
ax[0].set_title("Espera promedio")
ax[0].set_xlabel("Personas servidoras")
ax[0].set_ylabel("Minutos")

sns.boxplot(
    data=resultados, x="servidores", y="p90_espera",
    color="#A7F3D0", ax=ax[1]
)
ax[1].axhline(15, color="#DC2626", linestyle="--")
ax[1].set_title("Percentil 90 por jornada")
ax[1].set_xlabel("Personas servidoras")
ax[1].set_ylabel("Minutos")

sns.barplot(
    data=tabla, x="servidores", y="utilizacion",
    color="#F59E0B", ax=ax[2]
)
ax[2].set_title("Utilización promedio")
ax[2].set_xlabel("Personas servidoras")
ax[2].set_ylabel("Porcentaje")

plt.tight_layout()
plt.show()

display(tabla.round(2))

### Explicación sencilla de la comparación

La tabla reúne la evidencia principal. Un escenario puede tener una espera promedio baja, pero todavía presentar un percentil 90 alto. Otro puede tener una buena experiencia, pero una utilización demasiado baja.

Los criterios de 5 minutos de espera promedio y 15 minutos en el percentil 90 son metas didácticas para este ejercicio. En una organización real deben acordarse con el área, porque una solicitud urgente y una solicitud rutinaria no necesariamente tienen la misma promesa de atención.

La recomendación automática busca la alternativa mínima que cumple ambos criterios. Esto representa una política prudente: no usar más capacidad de la necesaria, pero tampoco aceptar una espera que el área considere excesiva.

### Interpretación detallada de resultados

La espera promedio responde cómo vive el servicio una persona típica. El percentil 90 responde qué ocurre con el grupo que enfrenta mayor demora. Conviene revisar ambos porque un promedio bajo puede ocultar casos extremos.

Los pendientes al cierre muestran continuidad operativa. Un valor positivo significa que parte del trabajo pasa al siguiente día y puede generar acumulación u horas extra.

La utilización muestra cuánto se ocupó la capacidad. Una persona al 95% puede parecer eficiente, pero tiene poca reserva para picos. Una utilización moderada puede ser deseable si protege el nivel de servicio.

Los criterios didácticos usados aquí son espera promedio menor a 5 minutos y percentil 90 menor a 15 minutos. No son una norma universal: deben sustituirse por el acuerdo real del área.

## 15. Acumulación durante la jornada

Se estima cuántas solicitudes están todavía dentro del sistema en cada minuto. Esta cantidad incluye solicitudes esperando y en servicio.

Una línea que crece continuamente indica que las llegadas superan la capacidad de salida durante buena parte de la jornada.

In [ ]:
def inventario_minuto(datos):
    minutos = np.arange(0, int(DURACION_JORNADA) + 1)
    valores = []

    for minuto in minutos:
        llegadas = (datos["llegada"] <= minuto).sum()
        salidas = (datos["salida"] <= minuto).sum()
        valores.append(llegadas - salidas)

    return pd.DataFrame({
        "minuto": minutos,
        "solicitudes_en_sistema": valores
    })

datos_1 = simular_jornada(1, semilla=SEMILLA_BASE)
datos_2 = simular_jornada(2, semilla=SEMILLA_BASE)

inv_1 = inventario_minuto(datos_1)
inv_2 = inventario_minuto(datos_2)

plt.figure(figsize=(14, 5))
plt.plot(inv_1["minuto"], inv_1["solicitudes_en_sistema"],
         color="#EF4444", label="1 persona")
plt.plot(inv_2["minuto"], inv_2["solicitudes_en_sistema"],
         color="#2563EB", label="2 personas")
plt.axvline(DURACION_JORNADA, color="black", linestyle="--",
            label="Cierre")
plt.title("Solicitudes no finalizadas durante la jornada")
plt.xlabel("Minutos")
plt.ylabel("Solicitudes en el sistema")
plt.legend()
plt.show()

### Explicación sencilla de las visualizaciones comparativas

La barra de espera promedio facilita una comparación rápida entre capacidades. El boxplot añade la variabilidad: muestra si el resultado es estable entre jornadas o si algunos días son mucho peores.

La utilización debe leerse junto con la espera. Si una persona tiene una utilización cercana a 100% y una espera elevada, el sistema está saturado. Si tres personas tienen una utilización cercana a 47% y una espera muy baja, existe capacidad de reserva, aunque puede ser más costosa.

La decisión no debe tomarse por el color o por la altura de una sola barra. Se deben revisar los tres indicadores y la prioridad del servicio.

### Interpretación de la acumulación

Comparar una y dos personas con la misma semilla permite atribuir la diferencia principalmente a la capacidad.

Si el inventario crece con una persona y se mantiene menor con dos, la segunda alternativa absorbe mejor la demanda presentada. Aun así, esta gráfica muestra una jornada; la decisión debe sustentarse en las 30 réplicas.

## 16. Sensibilidad ante demanda alta

Se prueban tres niveles:

- baja: una solicitud cada 8 minutos;
- base: una solicitud cada 6 minutos;
- alta: una solicitud cada 4.5 minutos.

La demanda alta representa campañas, fechas límite o temporadas. El objetivo es conocer si una alternativa funciona únicamente en un día normal o también conserva calidad durante presión.

In [ ]:
sensibilidad = []
niveles = {"Baja": 8.0, "Base": 6.0, "Alta": 4.5}

for nombre, media in niveles.items():
    for servidores in [1, 2, 3]:
        for replica in range(20):
            datos = simular_jornada(
                servidores,
                media_interarribo=media,
                semilla=SEMILLA_BASE + replica
            )
            fila = resumir_jornada(datos, servidores)
            fila["demanda"] = nombre
            sensibilidad.append(fila)

sensibilidad = pd.DataFrame(sensibilidad)

display(
    sensibilidad.groupby(["demanda", "servidores"])[
        ["espera_promedio", "p90_espera",
         "pendientes", "utilizacion_pct"]
    ].mean().round(2)
)

### Explicación sencilla de la acumulación

Esta gráfica cuenta las solicitudes que ya llegaron y aún no han terminado. Incluye personas en espera y personas que están siendo atendidas.

Una línea creciente significa que el sistema está recibiendo trabajo más rápido de lo que puede terminarlo. Una línea más estable significa que la capacidad está equilibrando mejor las llegadas.

Comparar una y dos personas con la misma semilla hace visible el efecto de agregar capacidad bajo la misma secuencia de demanda. Aun así, una sola trayectoria no sustituye el promedio de muchas réplicas.

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=sensibilidad,
    x="servidores",
    y="espera_promedio",
    hue="demanda",
    palette={"Baja": "#10B981", "Base": "#2563EB", "Alta": "#EF4444"},
    errorbar="sd"
)

plt.axhline(5, color="black", linestyle="--")
plt.title("Espera ante cambios en la demanda")
plt.xlabel("Personas servidoras")
plt.ylabel("Espera promedio (minutos)")
plt.legend(title="Demanda")
plt.tight_layout()
plt.show()

### Explicación sencilla del análisis de demanda

El promedio entre llegadas cambia de 8 a 6 y luego a 4.5 minutos. Al disminuir el intervalo, las solicitudes llegan más frecuentemente y la presión operativa aumenta.

La demanda alta representa situaciones comunes: fechas límite, campañas, periodos de inscripción o cierres administrativos. Probarlas es importante porque una solución que funciona en un día normal puede fallar en una temporada crítica.

Las barras de error muestran la variación entre réplicas. Si son grandes, la experiencia del servicio depende mucho de cómo se distribuyen las llegadas y los servicios durante el día.

### Interpretación de sensibilidad

Cuando el promedio entre llegadas baja, aumenta la presión sobre el sistema. Una alternativa que funciona en demanda base podría fallar en demanda alta.

Si el resultado indica que se necesita una capacidad adicional solo en demanda alta, pueden evaluarse medidas temporales: refuerzo en horas pico, citas, priorización, digitalización o simplificación del trámite.

La variabilidad entre réplicas, representada por las barras de error, indica el riesgo de que un día sea mucho peor que el promedio.

## 17. Recomendación, limitaciones y conclusiones

La recomendación debe elegir la menor capacidad que cumpla los objetivos de espera y mantenga pendientes bajos, dejando margen operativo.

Antes de usar el modelo en una decisión real, convendría:

- medir llegadas por hora;
- separar tipos de trámite;
- ajustar el servicio con datos históricos;
- modelar pausas, ausencias y prioridades;
- incorporar abandono de la cola;
- incluir costos de personal y de espera;
- validar con quienes operan el proceso.

### Conclusiones

1. SimPy representa de forma clara llegadas, colas, atención y salidas.
2. La simulación de eventos discretos es especialmente adecuada para procesos de servicio.
3. La capacidad modifica espera, pendientes y utilización.
4. El percentil 90 permite detectar una mala experiencia que el promedio puede ocultar.
5. Las réplicas muestran la variabilidad entre jornadas.
6. La sensibilidad revela si una solución es robusta ante temporadas de alta demanda.
7. El siguiente paso recomendado es sustituir los supuestos sintéticos por datos reales y recalibrar el notebook.

El modelo convierte la pregunta administrativa “¿cuántas personas necesitamos?” en una comparación cuantitativa, visual y reproducible.

In [ ]:
candidatos = tabla[
    (tabla["cumple_promedio"]) &
    (tabla["cumple_p90"])
]

for _, fila in tabla.iterrows():
    print(
        f"{int(fila['servidores'])} servidor(es): "
        f"espera={fila['espera_promedio']:.2f} min, "
        f"P90={fila['p90_espera']:.2f} min, "
        f"pendientes={fila['pendientes']:.2f}, "
        f"utilización={fila['utilizacion']:.1f}%"
    )

if len(candidatos):
    elegido = candidatos.sort_values("servidores").iloc[0]
    print(
        f"\nAlternativa mínima que cumple los criterios didácticos: "
        f"{int(elegido['servidores'])} servidor(es)."
    )
else:
    print("\nNinguna alternativa cumple ambos criterios.")

### Explicación sencilla de la salida ejecutiva

El texto automático resume los indicadores para facilitar una lectura rápida. No reemplaza el análisis: sirve como punto de partida para que la persona responsable revise la tabla y las gráficas.

Una recomendación operativa debe contestar tres preguntas: ¿se atiende con rapidez?, ¿se termina el trabajo?, y ¿se usa razonablemente el personal? La alternativa recomendada debe equilibrar esas tres dimensiones.

Si ninguna alternativa cumple, eso no significa únicamente que falte personal. También puede ser necesario rediseñar el trámite, distribuir citas, automatizar pasos o separar solicitudes complejas de solicitudes sencillas.

## 21. Revisión de realismo de los datos y de las unidades

Esta sección confirma que los datos sintéticos son adecuados para un centro de servicio administrativo.

### Unidades

- La jornada es de 480 minutos, equivalente a 8 horas.
- Las llegadas se expresan como minutos entre solicitudes.
- La atención se expresa como minutos por solicitud.
- La espera y el tiempo total también se expresan en minutos.
- La utilización se expresa como porcentaje.

### Magnitudes

Con una llegada promedio cada 6 minutos, se esperan aproximadamente 10 solicitudes por hora y 80 en una jornada. Un servicio promedio de 9 minutos permite cerca de 6.7 atenciones por hora por persona.

Por esa razón:

- una persona tiene capacidad media inferior a la demanda;
- dos personas tienen capacidad media superior a la demanda;
- tres personas ofrecen mayor margen, pero pueden tener menor utilización.

Estos valores son plausibles para solicitudes administrativas de complejidad baja o media. No representan una ventanilla que atienda consultas de pocos segundos ni un trámite especializado que tarde horas.

### Valores no negativos

Los tiempos de llegada, espera y servicio no pueden ser negativos. El modelo usa distribuciones que garantizan esto:

- la distribución exponencial produce tiempos entre llegadas mayores o iguales a cero;
- la distribución triangular queda entre 4 y 15 minutos;
- la espera se calcula como inicio menos llegada, y SimPy solo inicia un servicio después de que la solicitud llega.

### Pendientes

Los pendientes no son solicitudes con valores negativos ni errores de cálculo. Son solicitudes que sí llegaron durante la jornada, pero que no alcanzaron a completar el servicio antes del minuto 480. Este resultado es precisamente una señal de saturación o de falta de capacidad.

### Alcance de la realidad

Los datos son sintéticos. Son razonables como ejemplo didáctico, pero no deben presentarse como mediciones de una oficina específica. Para una decisión real se deben sustituir por datos de llegadas, tiempos de atención, horarios, pausas, prioridades y tipos de trámite observados.


## 22. Lectura de los resultados observados en esta ejecución

Con la semilla configurada en el notebook, la comparación base produjo aproximadamente los siguientes resultados promedio:

| Personas servidoras | Espera promedio | Percentil 90 | Pendientes | Utilización |
|---:|---:|---:|---:|---:|
| 1 | 87.36 min | 147.09 min | 26.63 | 98.0% |
| 2 | 4.21 min | 11.69 min | 1.70 | 71.2% |
| 3 | 0.71 min | 2.69 min | 1.43 | 47.0% |

### ¿Qué significa esto en palabras sencillas?

Con una persona, el centro recibe más trabajo del que puede procesar en promedio. La persona servidora permanece ocupada casi toda la jornada, pero eso no significa que el servicio sea bueno: la espera promedio supera una hora y el percentil 90 llega a más de dos horas. La utilización alta está acompañada de saturación.

Con dos personas, la espera promedio baja a cerca de 4 minutos y el percentil 90 es de aproximadamente 12 minutos. Esta alternativa cumple las metas didácticas usadas en el notebook y deja una utilización cercana a 71%, que representa un margen operativo razonable para absorber variación.

Con tres personas, la espera baja todavía más, pero la utilización promedio cae a cerca de 47%. Esta opción ofrece más holgura, pero puede ser excesiva para la demanda base si el costo del personal es importante.

### Advertencia importante sobre demanda alta

En la prueba de sensibilidad, con llegadas cada 4.5 minutos en promedio, dos personas ya no mantienen la misma calidad: la espera promedio observada fue cercana a 21 minutos y el percentil 90 superó 38 minutos. Tres personas redujeron la espera promedio a cerca de 2 minutos.

Por tanto, la recomendación debe expresarse con contexto:

- para un día base, dos personas son la alternativa mínima que cumple las metas didácticas;
- para temporadas de demanda alta, puede ser necesario un refuerzo temporal o una estrategia adicional;
- la decisión real debe incorporar costos, horarios, tipos de trámite y una promesa de servicio acordada.

Estos números corresponden a una ejecución reproducible del ejemplo, no a una medición de una oficina específica. Si se cambia la semilla o los parámetros, los valores cambiarán, pero la forma de interpretar las métricas seguirá siendo la misma.



## 23. Cómo leer una solicitud de principio a fin

Para entender la simulación, imaginemos una solicitud concreta:

1. La persona llega en el minuto 100.
2. Si existe una persona servidora libre, comienza casi inmediatamente.
3. Si todas están ocupadas, espera en la fila.
4. Supongamos que empieza en el minuto 112: su espera fue de 12 minutos.
5. Si su trámite tarda 8 minutos, termina en el minuto 120.
6. Su tiempo total en el sistema fue 20 minutos.

En este ejemplo:

- llegada = 100 minutos;
- inicio = 112 minutos;
- espera = 112 - 100 = 12 minutos;
- servicio = 8 minutos;
- tiempo total = 12 + 8 = 20 minutos.

La simulación repite esta lógica para todas las solicitudes. La cola no se inventa al final: aparece cuando la demanda y los tiempos de servicio hacen que no exista una persona libre en el instante de llegada.

Esta forma de pensar permite relacionar cada columna de la tabla con una experiencia real de una persona usuaria.



## 24. Diferencia entre espera, servicio y tiempo en el sistema

Estos tres conceptos suelen confundirse:

### Espera

Es el tiempo antes de comenzar la atención. Depende principalmente de cuántas solicitudes llegaron antes y de si las personas servidoras estaban ocupadas.

Una espera de 0 minutos no significa que el trámite haya sido instantáneo. Significa que la atención comenzó inmediatamente.

### Servicio

Es el tiempo durante el cual una persona servidora trabaja en el trámite. En este ejemplo siempre está entre 4 y 15 minutos porque se definió así la distribución triangular.

### Tiempo en el sistema

Es la experiencia completa desde la llegada hasta la salida:

tiempo en el sistema = espera + servicio

Por ejemplo, un trámite puede tardar 9 minutos en resolverse, pero la persona puede permanecer 40 minutos en el centro si pasó 31 minutos esperando. Para mejorar la experiencia, no basta con reducir el tiempo de atención: también hay que controlar la espera.



## 25. Interpretación sencilla de cada indicador

### Solicitudes generadas

Es la demanda que realmente apareció en la jornada simulada. No tiene que ser exactamente 80 porque 80 es un promedio esperado, no una cuota fija.

### Solicitudes atendidas

Son las que terminaron dentro de la jornada. Si hay menos atendidas que generadas, existe trabajo que no terminó a tiempo.

### Pendientes al cierre

Son las solicitudes que llegaron, pero que todavía no concluyeron al minuto 480. Un pendiente no significa que la solicitud se haya perdido; significa que requiere continuidad posterior.

### Espera promedio

Es la media de todas las esperas observadas. Es sencilla de comunicar, pero puede verse afectada por esperas extremas.

### Mediana de espera

Ordena las esperas y toma el valor central. Si la media es mucho mayor que la mediana, probablemente existen algunos casos muy demorados que elevan el promedio.

### Percentil 90

Es un indicador de protección al usuario. Si vale 12 minutos, aproximadamente 90 de cada 100 solicitudes esperan 12 minutos o menos y alrededor de 10 esperan más.

### Espera máxima

Muestra el peor caso de las réplicas resumidas. No debe usarse por sí sola para dimensionar, pero sirve para descubrir experiencias potencialmente inaceptables.

### Utilización

Indica qué parte de la capacidad teórica se usa. Una utilización elevada no siempre significa buen desempeño: si se acerca al 100%, cualquier llegada adicional o trámite largo puede crear una cola importante.



## 26. ¿Qué está diciendo la comparación de escenarios?

Los resultados observados muestran una diferencia clara:

- con 1 persona, la espera promedio es cercana a 87 minutos, el percentil 90 supera 147 minutos y quedan cerca de 27 solicitudes pendientes por jornada;
- con 2 personas, la espera promedio es cercana a 4 minutos, el percentil 90 es cercano a 12 minutos y quedan alrededor de 2 solicitudes pendientes;
- con 3 personas, la espera promedio es menor a 1 minuto y el percentil 90 es cercano a 3 minutos, pero la utilización baja a menos de 50%.

La conclusión sencilla es que una persona está sobrecargada, dos personas equilibran mejor la demanda base y tres personas ofrecen una reserva más amplia a cambio de utilizar menos intensamente el equipo.

Esto no significa que dos personas sean siempre la respuesta. Significa que, bajo los supuestos de este notebook y las metas didácticas elegidas, dos es la alternativa mínima que logra un servicio razonable en la demanda base.

La decisión cambia si cambia la prioridad:

- si el objetivo principal es reducir al mínimo la espera, tres personas son mejores;
- si el objetivo es equilibrar costo y servicio en un día normal, dos personas son más razonables;
- si se espera una temporada de alta demanda, se debe revisar el escenario de sensibilidad.



## 27. Cómo interpretar la gráfica de demanda alta

En el escenario de demanda alta, las solicitudes llegan cada 4.5 minutos en promedio, en lugar de cada 6. Esto equivale a aproximadamente 13.3 solicitudes por hora, frente a 10 solicitudes por hora en la demanda base.

Con dos personas, la espera observada aumenta a aproximadamente 21 minutos y el percentil 90 supera 38 minutos. Esto muestra que una capacidad que funciona en el promedio puede dejar de ser suficiente durante un pico.

Con tres personas, la espera se mantiene cercana a 2 minutos en esa prueba. La lectura administrativa es importante: quizá no sea necesario mantener tres personas todo el año, pero sí puede ser conveniente programar un refuerzo temporal en fechas críticas.

La gráfica no dice qué política debe adoptarse automáticamente. Ayuda a comparar opciones como:

- contratar o reasignar personal temporal;
- abrir una ventanilla adicional durante ciertas horas;
- atender trámites sencillos por un canal rápido;
- usar citas para distribuir las llegadas;
- digitalizar pasos que consumen tiempo.



## 28. Cómo explicar el resultado a una persona no técnica

Una explicación ejecutiva podría ser:

> Si una sola persona atiende este volumen de solicitudes, el equipo opera casi al límite y la espera se acumula. Con dos personas, la mayoría de las solicitudes se atiende en pocos minutos y la utilización permanece en un nivel razonable. En periodos de alta demanda, conviene contar con un refuerzo temporal o rediseñar el flujo.

Esta frase utiliza tres ideas fáciles de verificar en el notebook:

1. demanda aproximada de 80 solicitudes en 8 horas;
2. atención promedio de 9 minutos;
3. comparación de espera, pendientes y utilización entre capacidades.

La simulación sirve como una herramienta para conversar sobre decisiones. No reemplaza el conocimiento del personal ni la validación con datos reales.
